# 02 - Sliding Window Dataset & Time Horizon Formulation
### Deep Learning for Forecasting Quantum Dynamics – PXP Many-Body Model

Questo notebook definisce la **pipeline di windowing temporale** per convertire le traiettorie fisiche continue in coppie supervised $(X, Y)$ e affronta la formulazione della **SFIDA 1: Time Window Optimization**.

---

### 🎯 Obiettivi del Notebook:
1. **Formulazione Matematica del Windowing**: Definizione rigorosa di sequenza di input $X \in \mathbb{R}^{T_{in} \times 55}$ e orizzonte di previsione futuro $Y \in \mathbb{R}^{T_{out} \times 55}$.
2. **Griglia Controllata per la Sfida 1**:
   - $\text{Short Window}$: $T_{in} = 50$ step ($t = 1.0\text{ s}$)
   - $\text{Medium Window}$: $T_{in} = 100$ step ($t = 2.0\text{ s}$)
   - $\text{Long Window}$: $T_{in} = 200$ step ($t = 4.0\text{ s}$)
   - Orizzonte futuro target comune: $T_{out} = 100$ step ($t = 2.0\text{ s}$)
3. **Analisi dello Stride**: studio quantitativo del numero di campioni generati e bilanciamento della ridondanza per evitare overfitting.
4. **Visualizzazione Intuitiva del Windowing**: diagramma ad alta risoluzione $(\ge 300\text{ DPI})$ che mostra la suddivisione tra input passato e futuro target.
5. **Costruzione e Test dei PyTorch DataLoader**: integrazione con `src/utils/windowing.py` per servire batch efficienti e pronti all'uso per i modelli M1, M2, M3, M4.


In [ ]:
import os
import sys
import time
from pathlib import Path

# Setup del percorso al modulo src/
PROJECT_ROOT = Path(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import torch

from utils.seeding import set_seeds
from utils.data_loading import load_dataset
from utils.windowing import (
    WINDOW_CONFIGS,
    create_sliding_windows,
    QuantumWindowDataset,
    get_dataloaders
)

# Determinismo totale
set_seeds(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device di esecuzione: {device}")


## 1. Caricamento del Dataset e degli Indici di Split Ufficiali

Carichiamo le feature binarie (`data/trajectories.npy`) e gli indici di partizionamento congelati nello Step 1 (`data/split_indices.npz`).


In [ ]:
data_dir = PROJECT_ROOT / "data"

# Caricamento del tensore (400, 1001, 55)
features, time_grid = load_dataset(str(data_dir))

# Caricamento indici train/val/test
split_data = np.load(data_dir / "split_indices.npz")
split_indices = {
    "train_idx": split_data["train_idx"],
    "val_idx": split_data["val_idx"],
    "test_idx": split_data["test_idx"],
}

print(f"Dataset caricato: {features.shape} (400 traiettorie, 1001 istanti, 55 osservabili)")
print(f"Split caricato: {len(split_indices['train_idx'])} Train | {len(split_indices['val_idx'])} Val | {len(split_indices['test_idx'])} Test")


## 2. Formulazione della Griglia Sperimentale (Sfida 1)

Nella **Sfida 1**, l'obiettivo è identificare **la finestra temporale di osservazione minima $T_{in}$** che permette ai modelli di generalizzare accuratamente sul futuro $T_{out}$.

Definiamo le 3 configurazioni standard:

| Configurazione | $T_{in}$ (step) | $T_{in}$ (tempo fisico) | $T_{out}$ (step target) | Obiettivo Scientifico |
| :--- | :---: | :---: | :---: | :--- |
| **`short_window`** | $50$ | $1.0\text{ s}$ | $100$ ($2.0\text{ s}$) | **Sfida Estrema**: osservazione di frazione di periodo di oscillazione. |
| **`medium_window`** | $100$ | $2.0\text{ s}$ | $100$ ($2.0\text{ s}$) | Finestra bilanciata: cattura almeno un ciclo coerente completo. |
| **`long_window`** | $200$ | $4.0\text{ s}$ | $100$ ($2.0\text{ s}$) | Benchmark ad alta informazione: cattura la dinamica a lungo raggio. |


In [ ]:
for cfg in WINDOW_CONFIGS:
    in_t = cfg["input_len"] * 0.02
    out_t = cfg["output_len"] * 0.02
    print(f"Configurazione: {cfg['name']:<15} | T_in = {cfg['input_len']:>3} passi ({in_t:.2f} s) | T_out = {cfg['output_len']:>3} passi ({out_t:.2f} s)")


## 3. Analisi della Strategia di Stride

Lo `stride` $S$ determina di quanti passi temporali trasliamo la finestra scorrevole lungo la traiettoria di lunghezza $N_{points} = 1001$.

Il numero di campioni estratti da una singola traiettoria è:
$$N_{\text{windows}} = \left\lfloor \frac{N_{points} - (T_{in} + T_{out})}{S} \right\rfloor + 1$$

- Con `stride = 1`: due finestre consecutive si sovrappongono per il $99\%$ del loro contenuto, quadruplicando la memoria e rischiando overfitting su campioni quasi identici.
- Con `stride = 5`: riduciamo la ridondanza dell' $80\%$, mantenendo decine di migliaia di campioni ricchi e diversificati.
- Con `stride = 10` per Validation e Test: valutiamo i modelli su punti temporali equispaziati garantendo velocità ed equità di confronto.


In [ ]:
print("=== ANALISI CAMPIONI GENERATI (320 Traiettorie di Train) ===")
print(f"{'Configurazione':<16} | {'Stride=1':<12} | {'Stride=2':<12} | {'Stride=5':<12} | {'Stride=10':<12}")
print("-" * 72)

for cfg in WINDOW_CONFIGS:
    tot_win = cfg["input_len"] + cfg["output_len"]
    n_per_traj = lambda s: (1001 - tot_win) // s + 1
    
    s1 = n_per_traj(1) * 320
    s2 = n_per_traj(2) * 320
    s5 = n_per_traj(5) * 320
    s10 = n_per_traj(10) * 320
    
    print(f"{cfg['name']:<16} | {s1:<12} | {s2:<12} | {s5:<12} | {s10:<12}")


## 4. Visualizzazione Grafica dello Schema di Windowing

Mostriamo graficamente come una finestra $(X, Y)$ viene estratta da una traiettoria reale (Traiettoria 0, prima magnetizzazione $m_1(t)$).


In [ ]:
# Estraiamo un esempio con configurazione short_window (T_in = 50, T_out = 100)
cfg = WINDOW_CONFIGS[0]
in_len = cfg["input_len"]
out_len = cfg["output_len"]
tot_len = in_len + out_len

start_idx = 100  # Offset temporale di esempio

t_slice = time_grid[start_idx : start_idx + tot_len]
m1_slice = features[0, start_idx : start_idx + tot_len, 0]

plt.figure(figsize=(14, 5))

# Plot dell'intera sequenza
plt.plot(t_slice[:in_len], m1_slice[:in_len], color="#1f77b4", linewidth=2.5, label="Input Osservato ($X$: $T_{in} = 50$ passi)")
plt.plot(t_slice[in_len:], m1_slice[in_len:], color="#d62728", linewidth=2.5, linestyle="--", label="Target Futuro ($Y$: $T_{out} = 100$ passi)")

# Aree colorate di contesto
plt.axvspan(t_slice[0], t_slice[in_len - 1], color="#1f77b4", alpha=0.15, label="Finestra Passata (Input)")
plt.axvspan(t_slice[in_len], t_slice[-1], color="#d62728", alpha=0.15, label="Orizzonte di Rollout (Previsione)")

# Linea di separazione
plt.axvline(t_slice[in_len], color="black", linestyle=":", linewidth=1.5, label="Istante Presente ($t_{now}$)")

plt.title("Schema di Sliding Window: Input Passato $X$ vs Target Futuro $Y$ (Qubit 1)", fontsize=13)
plt.xlabel("Tempo Fisico $t$ (secondi)", fontsize=11)
plt.ylabel("Magnetizzazione $m_1(t)$", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right", framealpha=0.9)

fig_path = PROJECT_ROOT / "results" / "figures" / "windowing_scheme.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Grafico salvato in: {fig_path} (300 DPI)")


## 5. Costruzione e Benchmark dei PyTorch DataLoader

Testiamo la funzione integrata `get_dataloaders()` per la configurazione `short_window`.
Verifichiamo la forma dei batch e misuriamo il throughput di caricamento.


In [ ]:
# Creazione DataLoader con batch_size = 64
batch_size = 64

train_loader, val_loader, test_loader, selected_cfg = get_dataloaders(
    features=features,
    split_indices=split_indices,
    config_name="short_window",
    stride_train=5,
    stride_eval=10,
    batch_size=batch_size,
    num_workers=0
)

print(f"=== DATALOADER PRONTI ({selected_cfg['name']}) ===")
print(f"Batch di Training:   {len(train_loader)} (Totale campioni: {len(train_loader.dataset)})")
print(f"Batch di Validation: {len(val_loader)} (Totale campioni: {len(val_loader.dataset)})")
print(f"Batch di Test:       {len(test_loader)} (Totale campioni: {len(test_loader.dataset)})")

# Ispezione del primo batch
first_x, first_y = next(iter(train_loader))
print(f"\nForma Input X:  {first_x.shape} -> (Batch, T_in, Features) [dtype: {first_x.dtype}]")
print(f"Forma Target Y: {first_y.shape} -> (Batch, T_out, Features) [dtype: {first_y.dtype}]")

assert first_x.shape == (batch_size, selected_cfg["input_len"], 55)
assert first_y.shape == (batch_size, selected_cfg["output_len"], 55)
print("Sanity Check Dimensionale del Batch: SUPERATO!")


### 5.1 Test di Velocità del DataLoader

Misuriamo il tempo necessario per iterare attraverso 100 batch casuali.


In [ ]:
start_time = time.time()
num_batches_test = min(100, len(train_loader))

for i, (bx, by) in enumerate(train_loader):
    if i >= num_batches_test:
        break

elapsed = time.time() - start_time
throughput = (num_batches_test * batch_size) / elapsed

print(f"Iterati {num_batches_test} batch in {elapsed:.3f} s")
print(f"Throughput: {throughput:.0f} campioni/secondo")
print("La pipeline in memoria è ultra-veloce e non costituirà un collo di bottiglia durante il training.")


## 6. Conclusioni e Predisposizione per i Modelli

In questo notebook abbiamo:
1. Formalizzato la griglia comune di finestre per la **Sfida 1** (`short_window`, `medium_window`, `long_window`).
2. Definito la strategia di stride ottimale (`stride_train=5`, `stride_eval=10`).
3. Creato e testato i `DataLoader` PyTorch veloci e standardizzati.
4. Esportato il grafico illustrativo in `results/figures/windowing_scheme.png`.

---

### 🚀 Prossimo Notebook: `03_model_m1_lstm_direct.ipynb`
Con i DataLoader pronti, implementeremo il primo modello: **M1 (LSTM Multi-Step Diretto)**, stabilendo la **baseline iniziale** delle prestazioni predittive sulla catena di 10 qubit.
